# H2: Vigor Dynamics Across the Threat Imminence Continuum

**Claim:** Threat modulates motor vigor independently of choice, with distinct dynamics across the predatory imminence continuum.

- **H2a:** Threat increases pressing rate within cookie type
- **H2b:** Predator encounter triggers a threat-independent motor spike
- **H2c:** Threat and encounter have distinct temporal signatures (GAM)
- **H2d:** Stable individual differences in pre-encounter threat-vigor modulation

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, ast, os
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix
from scipy.stats import pearsonr, ttest_rel, ttest_1samp, chi2, linregress
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

# Find repo root
REPO_ROOT = Path(os.getcwd())
for _ in range(5):
    if (REPO_ROOT / '.git').exists(): break
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data/exploratory_350/processed/stage5_filtered_data_20260320_191950"
RESULTS_DIR = REPO_ROOT / "results/stats"
EXCLUDE = [154, 197, 208]
BIN = 0.2; SMOOTH = 3; K_SPLINE = 10

def smooth(s):
    return pd.Series(s).rolling(SMOOTH, center=True, min_periods=1).mean().values

print(f"Repo root: {REPO_ROOT}")

# Load precomputed vigor metrics if available
vigor_path = RESULTS_DIR / "vigor_analysis/vigor_metrics.csv"
if vigor_path.exists():
    metrics = pd.read_csv(vigor_path)
    print(f"Loaded precomputed metrics: {len(metrics)} rows")
else:
    print("No precomputed metrics — will compute from raw data")
    metrics = None

# Load raw data for timecourse analyses
beh = pd.read_csv(DATA_DIR / "behavior_rich.csv", low_memory=False)
beh = beh[~beh['subj'].isin(EXCLUDE)]
beh['T_round'] = beh['threat'].round(1)
beh['is_heavy'] = (beh['trialCookie_weight'] == 3.0).astype(int)
beh['enc_t'] = pd.to_numeric(beh['encounterTime'], errors='coerce')
beh['is_attack'] = beh['isAttackTrial'].astype(int)

print(f"Trials: {len(beh)}, Subjects: {beh['subj'].nunique()}")

## H2a: Threat increases pressing rate within cookie type

**Test:** Paired t-test, T=0.9 vs T=0.1, within heavy and within light cookies separately.
**Threshold:** p < .01 for both.

In [ ]:
# Use precomputed full-trial metrics
full = metrics[metrics['epoch'] == 'full'] if metrics is not None else None

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, cookie, title in [(axes[0], 1, 'Heavy cookies (req=0.9)'),
                           (axes[1], 0, 'Light cookies (req=0.4)')]:
    sub = full[full['cookie'] == cookie]
    subj_means = sub.groupby(['subj', 'T_round'])['norm_rate'].mean().reset_index()
    cond = subj_means.groupby('T_round')['norm_rate'].agg(['mean', 'sem']).reset_index()

    colors = {0.1: '#2196F3', 0.5: '#9E9E9E', 0.9: '#F44336'}
    bars = ax.bar(range(3), cond['mean'], yerr=cond['sem']*1.96,
                  color=[colors[t] for t in cond['T_round']],
                  edgecolor='black', linewidth=0.5, capsize=5, alpha=0.8)
    ax.set_xticks(range(3))
    ax.set_xticklabels(['T=0.1', 'T=0.5', 'T=0.9'])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predator probability')

axes[0].set_ylabel('Normalized press rate\n(median IPI⁻¹ / calibMax)')
plt.suptitle('H2a: Threat increases vigor within cookie type', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Stats
print("H2a Results:")
print(f"{'Cookie':<8} {'M(T=.1)':>10} {'M(T=.9)':>10} {'Δ':>10} {'t':>8} {'p':>12} {'d':>8} {'Pass?':>6}")
print("-" * 72)
for cookie, label in [(1, 'Heavy'), (0, 'Light')]:
    sub = full[full['cookie'] == cookie]
    lo = sub[sub['T_round'] == 0.1].groupby('subj')['norm_rate'].mean()
    hi = sub[sub['T_round'] == 0.9].groupby('subj')['norm_rate'].mean()
    shared = sorted(set(lo.index) & set(hi.index))
    diff = hi.loc[shared].values - lo.loc[shared].values
    t, p = ttest_rel(hi.loc[shared], lo.loc[shared])
    d = diff.mean() / diff.std()
    passed = "✓" if p < 0.01 else "✗"
    print(f"{label:<8} {lo.loc[shared].mean():>10.4f} {hi.loc[shared].mean():>10.4f} "
          f"{diff.mean():>+10.4f} {t:>8.2f} {p:>12.2e} {d:>+8.3f} {passed:>6}")

## H2b: Predator encounter triggers a threat-independent motor spike

**Test 1:** Encounter spike = attack minus non-attack press rate in reactive epoch. One-sample t-test vs zero.
**Test 2:** Threat independence = paired t-test comparing spike at T=0.9 vs T=0.1.
**Threshold:** Spike p < .001; threat modulation p > .05.

In [ ]:
# Encounter spike from epoch metrics
reactive = metrics[metrics['epoch'] == 'reactive']

s_a = reactive[reactive['is_attack']==1].groupby('subj')['norm_rate'].mean()
s_n = reactive[reactive['is_attack']==0].groupby('subj')['norm_rate'].mean()
shared = sorted(set(s_a.index) & set(s_n.index))
spike = s_a.loc[shared] - s_n.loc[shared]

t_spike, p_spike = ttest_1samp(spike, 0)
d_spike = spike.mean() / spike.std()

print("H2b Test 1 — Encounter spike:")
print(f"  Mean spike: {spike.mean():+.4f}")
print(f"  t = {t_spike:.2f}, p = {p_spike:.2e}, d = {d_spike:+.3f}")
print(f"  {(spike>0).mean()*100:.0f}% of subjects positive")
print(f"  PASS: {'✓' if p_spike < 0.001 else '✗'} (threshold: p < .001)")

# Threat modulation
print("\nH2b Test 2 — Threat independence:")
spikes_by_T = {}
for T in [0.1, 0.5, 0.9]:
    sa = reactive[(reactive['is_attack']==1)&(reactive['T_round']==T)].groupby('subj')['norm_rate'].mean()
    sn = reactive[(reactive['is_attack']==0)&(reactive['T_round']==T)].groupby('subj')['norm_rate'].mean()
    sh = sorted(set(sa.index) & set(sn.index))
    spikes_by_T[T] = sa.loc[sh] - sn.loc[sh]
    print(f"  T={T}: spike = {spikes_by_T[T].mean():+.4f}")

sh_all = sorted(set(spikes_by_T[0.1].index) & set(spikes_by_T[0.9].index))
t_mod, p_mod = ttest_rel(spikes_by_T[0.9].loc[sh_all], spikes_by_T[0.1].loc[sh_all])
print(f"\n  T=0.9 vs T=0.1: t = {t_mod:.2f}, p = {p_mod:.4f}")
print(f"  PASS (threat-independent): {'✓' if p_mod > 0.05 else '✗'} (threshold: p > .05)")

# Figure: encounter-aligned timecourse
print("\nComputing encounter-aligned timecourse from raw keypresses...")
T_WIN = np.arange(-2.0, 5.0, BIN)
enc_recs = []
for _, row in beh.iterrows():
    try:
        pt = np.array(ast.literal_eval(str(row['alignedEffortRate'])), dtype=float)
        if len(pt) < 5: continue
        cal = row['calibrationMax']; enc = row['enc_t']
        if cal <= 0 or pd.isna(enc): continue
        ipis = np.diff(pt); midpoints = (pt[:-1]+pt[1:])/2
        rates = np.where(ipis > 0.01, (1.0/ipis)/cal, np.nan)
        enc_mid = midpoints - enc
        for t0 in T_WIN:
            mask = (enc_mid >= t0) & (enc_mid < t0 + BIN)
            v = rates[mask]; v = v[~np.isnan(v)]
            if len(v) >= 1:
                enc_recs.append(dict(subj=row['subj'], t_bin=round(t0+BIN/2,2),
                                    rate=np.median(v), cookie=row['is_heavy'],
                                    is_attack=row['is_attack']))
    except: pass
enc_df = pd.DataFrame(enc_recs)

# Cookie-center
cm = enc_df.groupby(['cookie','t_bin'])['rate'].transform('mean')
gm = enc_df.groupby('t_bin')['rate'].transform('mean')
enc_df['rate_cc'] = enc_df['rate'] - cm + gm

# Per-subject attack-noattack difference
subj_ts = enc_df.groupby(['subj','is_attack','t_bin'])['rate_cc'].mean().reset_index()
att = subj_ts[subj_ts['is_attack']==1].rename(columns={'rate_cc':'att'}).drop(columns='is_attack')
non = subj_ts[subj_ts['is_attack']==0].rename(columns={'rate_cc':'non'}).drop(columns='is_attack')
diff_ts = att.merge(non, on=['subj','t_bin'])
diff_ts['effect'] = diff_ts['att'] - diff_ts['non']
panel = diff_ts.groupby('t_bin')['effect'].agg(['mean','sem']).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
m = smooth(panel['mean'].values); s = smooth(panel['sem'].values)
ax.fill_between([0, 5], -0.01, max(m)+0.02, color='#FFEBEE', alpha=0.4, zorder=0)
ax.plot(panel['t_bin'], m, '#F44336', lw=2.5)
ax.fill_between(panel['t_bin'], m-1.96*s, m+1.96*s, color='#F44336', alpha=0.2)
ax.axhline(0, color='gray', ls='-', lw=0.8, alpha=0.5)
ax.axvline(0, color='gray', ls='--', lw=1.5, alpha=0.5)
ax.set_xlabel('Time from encounter (s)', fontsize=12)
ax.set_ylabel('Δ Press rate (attack − no attack)', fontsize=11)
ax.set_title('H2b: Encounter effect (within-subject, cookie-controlled)', fontweight='bold')
peak = panel.loc[panel['mean'].idxmax()]
ax.annotate(f'Peak: +{peak["mean"]:.3f}', xy=(peak['t_bin'], peak['mean']),
            xytext=(peak['t_bin']+0.5, peak['mean']+0.005), fontsize=9, color='#F44336')
plt.tight_layout()
plt.show()

## H2c: Threat and encounter have distinct temporal signatures

**Test:** GAM (spline basis + mixed model) with LRT for smooth-by-condition interactions.
**Threshold:** p < .01 for both encounter and threat LRTs.

In [ ]:
# Per-subject per-bin means (reduces data for GAM fitting)
gam_df = enc_df.groupby(['subj','t_bin','is_attack','cookie']).agg(
    rate=('rate','mean'), threat=('threat' if 'threat' in enc_df.columns else 'rate','mean')
).reset_index()

# Re-extract threat from the enc_df if missing
if 'threat' not in enc_df.columns:
    # Get threat from beh
    trial_threat = beh[['subj','trial','T_round']].drop_duplicates()
    # Use the enc_recs which have T_round... we need to rebuild
    # Simpler: add threat to enc_df from the raw records
    pass

# Actually rebuild gam_df from enc_recs which doesn't have threat
# Let's add it by re-extracting
enc_df2 = pd.DataFrame(enc_recs)  # has subj, t_bin, rate, cookie, is_attack but no threat
# For the GAM we need threat — aggregate without it and add as a separate model

# Build spline basis
spline_mat = dmatrix(f'cr(t_bin, df={K_SPLINE}) - 1', gam_df, return_type='dataframe')
spl_names = [f'spl{i}' for i in range(K_SPLINE)]
for i, col in enumerate(spline_mat.columns):
    gam_df[spl_names[i]] = spline_mat[col].values

# Interaction: spline × is_attack
atk_names = [f'a{i}' for i in range(K_SPLINE)]
for i in range(K_SPLINE):
    gam_df[atk_names[i]] = gam_df[spl_names[i]] * gam_df['is_attack']

spl_str = ' + '.join(spl_names)
atk_str = ' + '.join(atk_names)

# ENCOUNTER GAM
f_base = f'rate ~ {spl_str} + cookie'
f_full = f'rate ~ {spl_str} + {atk_str} + cookie'

print("Fitting GAMs...")
m_base = smf.mixedlm(f_base, gam_df, groups=gam_df['subj']).fit(reml=False)
m_full = smf.mixedlm(f_full, gam_df, groups=gam_df['subj']).fit(reml=False)

lr_enc = 2 * (m_full.llf - m_base.llf)
p_enc = 1 - chi2.cdf(lr_enc, K_SPLINE)

print(f"\nH2c — Encounter temporal signature:")
print(f"  LRT: χ² = {lr_enc:.1f}, df = {K_SPLINE}, p = {p_enc:.2e}")
print(f"  PASS: {'✓' if p_enc < 0.01 else '✗'} (threshold: p < .01)")
print(f"  Cookie: β = {m_full.fe_params['cookie']:.4f}, p = {m_full.pvalues['cookie']:.2e}")

# THREAT GAM — need threat in the data
# Re-extract with threat
enc_recs2 = []
for _, row in beh.iterrows():
    try:
        pt = np.array(ast.literal_eval(str(row['alignedEffortRate'])), dtype=float)
        if len(pt) < 5: continue
        cal = row['calibrationMax']; enc = row['enc_t']
        if cal <= 0 or pd.isna(enc): continue
        ipis = np.diff(pt); midpoints = (pt[:-1]+pt[1:])/2
        rates = np.where(ipis > 0.01, (1.0/ipis)/cal, np.nan)
        enc_mid = midpoints - enc
        for t0 in T_WIN:
            mask = (enc_mid >= t0) & (enc_mid < t0 + BIN)
            v = rates[mask]; v = v[~np.isnan(v)]
            if len(v) >= 1:
                enc_recs2.append(dict(subj=row['subj'], t_bin=round(t0+BIN/2,2),
                                     rate=np.median(v), cookie=float(row['is_heavy']),
                                     threat=row['T_round'], is_attack=float(row['is_attack'])))
    except: pass

thr_df = pd.DataFrame(enc_recs2)
thr_df = thr_df.groupby(['subj','t_bin','is_attack','cookie']).agg(
    rate=('rate','mean'), threat=('threat','mean')).reset_index()
thr_df['T_round'] = thr_df['threat'].round(1)
thr_df['T09'] = (thr_df['T_round'] == 0.9).astype(float)
thr_df['T05'] = (thr_df['T_round'] == 0.5).astype(float)

spline_mat2 = dmatrix(f'cr(t_bin, df={K_SPLINE}) - 1', thr_df, return_type='dataframe')
for i, col in enumerate(spline_mat2.columns):
    thr_df[spl_names[i]] = spline_mat2[col].values

t09_names = [f't09_{i}' for i in range(K_SPLINE)]
t05_names = [f't05_{i}' for i in range(K_SPLINE)]
for i in range(K_SPLINE):
    thr_df[t09_names[i]] = thr_df[spl_names[i]] * thr_df['T09']
    thr_df[t05_names[i]] = thr_df[spl_names[i]] * thr_df['T05']

t09_str = ' + '.join(t09_names)
t05_str = ' + '.join(t05_names)

f_base2 = f'rate ~ {spl_str} + cookie + is_attack'
f_full2 = f'rate ~ {spl_str} + {t09_str} + {t05_str} + cookie + is_attack'

m_base2 = smf.mixedlm(f_base2, thr_df, groups=thr_df['subj']).fit(reml=False)
m_full2 = smf.mixedlm(f_full2, thr_df, groups=thr_df['subj']).fit(reml=False)

lr_thr = 2 * (m_full2.llf - m_base2.llf)
p_thr = 1 - chi2.cdf(lr_thr, 2 * K_SPLINE)

print(f"\nH2c — Threat temporal signature:")
print(f"  LRT: χ² = {lr_thr:.1f}, df = {2*K_SPLINE}, p = {p_thr:.2e}")
print(f"  PASS: {'✓' if p_thr < 0.01 else '✗'} (threshold: p < .01)")

## H2d: Stable individual differences in pre-encounter threat-vigor modulation

**Test:** Per-subject anticipatory threat-vigor slope, cross-block stability.
**Threshold:** Mean cross-block r > 0.20.

In [ ]:
# Per-subject anticipatory threat-vigor slope
antic = metrics[metrics['epoch'] == 'anticipatory'] if metrics is not None else None

# Overall slopes (controlling for cookie)
slopes = []
for s, sdf in antic.groupby('subj'):
    if len(sdf) >= 10:
        X = sm.add_constant(sdf[['T_round', 'cookie']].values)
        y = sdf['norm_rate'].values
        try:
            ols = sm.OLS(y, X).fit()
            slopes.append({'subj': s, 'threat_slope': ols.params[1]})
        except: pass

slope_df = pd.DataFrame(slopes)
print(f"H2d — Individual threat-vigor slopes: {len(slope_df)} subjects")
print(f"  Mean slope: {slope_df['threat_slope'].mean():.4f}")
print(f"  SD: {slope_df['threat_slope'].std():.4f}")
print(f"  % positive: {(slope_df['threat_slope'] > 0).mean()*100:.0f}%")

# Cross-block stability
antic['block'] = (antic['trial'] // 27 + 1).astype(int)

block_slopes = {}
for block in [1, 2, 3]:
    block_s = []
    for s, sdf in antic[antic['block'] == block].groupby('subj'):
        if len(sdf) >= 3:
            X = sm.add_constant(sdf[['T_round', 'cookie']].values)
            y = sdf['norm_rate'].values
            try:
                ols = sm.OLS(y, X).fit()
                block_s.append({'subj': s, 'slope': ols.params[1]})
            except: pass
    block_slopes[block] = pd.DataFrame(block_s).set_index('subj')

# Cross-block correlations
print(f"\n  Cross-block stability:")
cross_rs = []
for b1, b2 in [(1,2), (1,3), (2,3)]:
    shared = sorted(set(block_slopes[b1].index) & set(block_slopes[b2].index))
    if len(shared) > 30:
        r, p = pearsonr(block_slopes[b1].loc[shared, 'slope'], block_slopes[b2].loc[shared, 'slope'])
        cross_rs.append(r)
        print(f"    Block {b1}-{b2}: r = {r:.3f} (N={len(shared)})")

mean_r = np.mean(cross_rs) if cross_rs else 0
print(f"\n  Mean cross-block r: {mean_r:.3f}")
print(f"  PASS: {'✓' if mean_r > 0.20 else '✗'} (threshold: mean r > 0.20)")

# Figure: slope distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.hist(slope_df['threat_slope'], bins=30, color='#9C27B0', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='gray', ls='--', lw=1)
ax.axvline(slope_df['threat_slope'].mean(), color='red', ls='-', lw=2, label=f"Mean={slope_df['threat_slope'].mean():.3f}")
ax.set_xlabel('Threat-vigor slope (anticipatory)')
ax.set_ylabel('Count')
ax.set_title('H2d: Individual threat-vigor slopes', fontweight='bold')
ax.legend(frameon=False)

ax = axes[1]
if len(cross_rs) >= 2:
    shared12 = sorted(set(block_slopes[1].index) & set(block_slopes[2].index))
    ax.scatter(block_slopes[1].loc[shared12, 'slope'], block_slopes[2].loc[shared12, 'slope'],
               s=15, alpha=0.4, color='#9C27B0')
    m, b = np.polyfit(block_slopes[1].loc[shared12, 'slope'], block_slopes[2].loc[shared12, 'slope'], 1)
    x = np.linspace(block_slopes[1].loc[shared12, 'slope'].min(), block_slopes[1].loc[shared12, 'slope'].max(), 100)
    ax.plot(x, m*x+b, 'k-', lw=2)
    ax.set_xlabel('Block 1 threat-vigor slope')
    ax.set_ylabel('Block 2 threat-vigor slope')
    ax.set_title(f'Cross-block stability (r={cross_rs[0]:.3f})', fontweight='bold')

plt.tight_layout()
plt.show()

## Summary

| Hypothesis | Test | Threshold | Result |
|-----------|------|-----------|--------|
| **H2a** | Paired t, T=0.9 vs T=0.1 within cookie | p < .01 both | |
| **H2b.1** | Encounter spike t-test vs 0 | p < .001 | |
| **H2b.2** | Spike threat modulation | p > .05 | |
| **H2c.1** | GAM LRT encounter shape | p < .01 | |
| **H2c.2** | GAM LRT threat shape | p < .01 | |
| **H2d** | Cross-block slope stability | mean r > .20 | |